In [27]:
import os
import re
import glob
import pandas as pd
from functools import reduce

def merge_diff_datasets(DIR):
    # ============================================================
    # Directory containing CSV result files
    # ============================================================
    #DIR = "m_2"   # <-- change this

    # ============================================================
    # Regex patterns
    # ============================================================

    # Extract lambda from filename
    lambda_pattern = re.compile(r"lambdda-([0-9.]+)\.csv")

    # Extract class (m, n, sigma) from instance filename
    # Example:
    # mglcs_10_100_4_1.txt.out
    # -> (10, 100, 4)
    instance_pattern = re.compile(
        r"mglcs_(\d+)_(\d+)_(\d+)_\d+\.txt\.out"
    )

    # ============================================================
    # Store aggregated dataframes
    # ============================================================
    dfs = []

    # ============================================================
    # Process each CSV file
    # ============================================================
    for filepath in sorted(glob.glob(os.path.join(DIR, "*.csv"))):

        filename = os.path.basename(filepath)
        #print(filename)
        # --------------------------------------------------------
        # Extract lambda value
        # --------------------------------------------------------
        lambda_match = lambda_pattern.search(filename)

        if not lambda_match:
            print(f"Skipping file (lambda not found): {filename}")
            continue

        lambda_value = lambda_match.group(1)

        print(f"Processing lambda = {lambda_value}")

        # --------------------------------------------------------
        # Read CSV
        # --------------------------------------------------------
        df = pd.read_csv(filepath)

        # --------------------------------------------------------
        # Extract (m, n, sigma)
        # --------------------------------------------------------
        extracted = df["file"].str.extract(instance_pattern)

        df["m"] = extracted[0].astype(int)
        df["n"] = extracted[1].astype(int)
        df["sigma"] = extracted[2].astype(int)

        # --------------------------------------------------------
        # Aggregate by graph class
        # --------------------------------------------------------
        grouped = (
            df.groupby(["m", "n", "sigma"])["quality"]
            .mean()
            .reset_index()
        )

        # Rename quality column to lambda value
        grouped = grouped.rename(columns={"quality": lambda_value})

        dfs.append(grouped)
    return dfs

In [28]:
# ============================================================
# Merge all aggregated dataframes
# ============================================================

def merge_dfs(dfs):
    
    merged_df = reduce(
        lambda left, right: pd.merge(
            left,
            right,
            on=["m", "n", "sigma"],
            how="outer"
        ),
        dfs
    )

    # ============================================================
    # Sort rows
    # ============================================================
    merged_df = merged_df.sort_values(
        by=["m", "n", "sigma"]
    ).reset_index(drop=True)

    # ============================================================
    # Save result
    # ============================================================
    #output_file = "aggregated_lambda_results.csv"

    #merged_df.to_csv(output_file, index=False)

    #print("\nFinal merged dataframe:")
    #print(merged_df)
 
    #print(f"\nSaved to: {output_file}")
    return merged_df
    

In [29]:
dfs_m2 = merge_diff_datasets("m_2")
dfs_m3 = merge_diff_datasets("m_3")
dfs_m5 = merge_diff_datasets("m_5")
dfs_m10 = merge_diff_datasets("m_10")

## merged results:
merged_df_m2 = merge_dfs(dfs_m2)
merged_df_m3 = merge_dfs(dfs_m3)
merged_df_m5 = merge_dfs(dfs_m5)
merged_df_m10 = merge_dfs(dfs_m10)

##filter out acc. to m:

merged_df_m2_filter = merged_df_m2[merged_df_m2["m"] == 2]
merged_df_m3_filter = merged_df_m3[merged_df_m3["m"] == 3]
merged_df_m5_filter = merged_df_m5[merged_df_m5["m"] == 5]
merged_df_m10_filter = merged_df_m10[merged_df_m10["m"] == 10]


Processing lambda = 0.00
Processing lambda = 0.05
Processing lambda = 0.1
Processing lambda = 0.15
Processing lambda = 0.2
Processing lambda = 0.25
Processing lambda = 0.3
Processing lambda = 0.35
Processing lambda = 0.4
Processing lambda = 0.45
Processing lambda = 0.5
Processing lambda = 0.55
Processing lambda = 0.60
Processing lambda = 0.65
Processing lambda = 0.70
Processing lambda = 0.75
Processing lambda = 0.80
Processing lambda = 0.85
Processing lambda = 0.90
Processing lambda = 0.95
Processing lambda = 1.0
Processing lambda = 0.00
Processing lambda = 0.05
Processing lambda = 0.10
Processing lambda = 0.15
Processing lambda = 0.20
Processing lambda = 0.25
Processing lambda = 0.30
Processing lambda = 0.35
Processing lambda = 0.40
Processing lambda = 0.45
Processing lambda = 0.50
Processing lambda = 0.55
Processing lambda = 0.60
Processing lambda = 0.65
Processing lambda = 0.70
Processing lambda = 0.75
Processing lambda = 0.80
Processing lambda = 0.85
Processing lambda = 0.90
Proces

In [30]:
merged_df_m2_filter["0.60"]  #lambda = 0.6 (for m=2)


0     37.7
1     30.1
2     72.9
3     62.1
4    140.7
5    125.8
6    299.5
7    310.9
Name: 0.60, dtype: float64

In [31]:
merged_df_m3_filter["0.50"]
#merged_df_m10_filter.mean()

8      31.2
9      22.9
10     58.5
11     48.7
12    106.9
13     98.0
14    157.2
15    238.3
Name: 0.50, dtype: float64

In [32]:
merged_df_m5_filter["0.35"] # lambda = 0.35 (for m=5)


16    20.5
17    15.3
18    32.8
19    27.9
20    46.6
21    42.4
22    45.6
23    56.7
Name: 0.35, dtype: float64

In [33]:
merged_df_m10_filter["0.15"] # lambda = 0.15 (for m=10)

24    11.6
25     7.5
26    14.4
27     9.9
28    14.8
29     9.5
30    14.6
31    10.1
Name: 0.15, dtype: float64

In [36]:
merged_df_m2_filter.mean()

m          2.0000
n        212.5000
sigma      3.0000
0.00     124.6375
0.05     126.3375
0.1      132.3125
0.15     131.8375
0.2      131.8375
0.25     133.4625
0.3      133.4375
0.35     134.1500
0.4      134.5000
0.45     134.6625
0.5      133.9000
0.55     134.1875
0.60     134.9625
0.65     134.4125
0.70     134.2375
0.75     132.4875
0.80     133.1625
0.85     132.0750
0.90     131.8500
0.95     132.0125
1.0      131.2125
dtype: float64